# Preprocessing & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Shape:', df.shape)
print('Duplicates:', df.duplicated().sum())

In [ ]:
# TotalCharges is object — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges:', df['TotalCharges'].isnull().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
# Drop customerID — not a feature
df.drop('customerID', axis=1, inplace=True)

# Encode binary Yes/No cols
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for c in binary_cols:
    df[c] = (df[c] == 'Yes').astype(int)

# Encode multi-value categoricals
cat_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# SeniorCitizen is already 0/1

print('Shape after encoding:', df.shape)
df.head()

In [ ]:
# Split
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Scale numericals
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
X_train.head()

In [ ]:
# Save basic feature set for before/after comparison
X_train_basic = X_train.copy()
X_test_basic = X_test.copy()

# --- Feature Engineering on X_train / X_test ---
def engineer_features(df):
    df = df.copy()
    # 1. Charge per month (average billing rate)
    df['charge_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)
    # 2. Tenure x MonthlyCharges (engagement proxy)
    df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
    # 3. Monthly deviation from average
    df['monthly_minus_avg'] = df['MonthlyCharges'] - (df['TotalCharges'] / (df['tenure'] + 1))
    # 4. Service binary flags from one-hot encoded Yes columns
    for src, dst in [('OnlineSecurity_Yes', 'OnlineSecurity_bin'),
                     ('OnlineBackup_Yes', 'OnlineBackup_bin'),
                     ('DeviceProtection_Yes', 'DeviceProtection_bin'),
                     ('TechSupport_Yes', 'TechSupport_bin'),
                     ('StreamingTV_Yes', 'StreamingTV_bin'),
                     ('StreamingMovies_Yes', 'StreamingMovies_bin')]:
        if src in df.columns:
            df[dst] = df[src].astype(int)
    # 5. Aggregate service features
    svc_cols = [c for c in df.columns if c.endswith('_bin')]
    df['services_count'] = df[svc_cols].sum(axis=1)
    df['no_services'] = (df['services_count'] == 0).astype(int)
    has_streaming = pd.Series(0, index=df.index)
    for c in ['StreamingTV_bin', 'StreamingMovies_bin']:
        if c in df.columns:
            has_streaming = has_streaming + df[c]
    df['has_streaming'] = (has_streaming > 0).astype(int)
    # 6. High-risk interaction features
    is_fiber = df['InternetService_Fiber optic'] if 'InternetService_Fiber optic' in df.columns else pd.Series(0, index=df.index)
    no_support = pd.Series(0, index=df.index)
    if 'TechSupport_Yes' in df.columns:
        no_support = (df['TechSupport_Yes'] == 0).astype(int)
    no_security = pd.Series(0, index=df.index)
    if 'OnlineSecurity_Yes' in df.columns:
        no_security = (df['OnlineSecurity_Yes'] == 0).astype(int)
    is_m2m = pd.Series(1, index=df.index)  # default: month-to-month (dropped reference)
    if 'Contract_One year' in df.columns:
        is_m2m = is_m2m - df['Contract_One year']
    if 'Contract_Two year' in df.columns:
        is_m2m = is_m2m - df['Contract_Two year']
    df['fiber_no_support'] = (is_fiber * no_support).astype(int)
    df['fiber_no_security'] = (is_fiber * no_security).astype(int)
    df['month_to_month_fiber'] = (is_m2m * is_fiber).astype(int)
    # 7. Tenure-based
    df['new_customer'] = (df['tenure'] <= 6).astype(int)
    # long_contract: 0=Month-to-month, 1=One year, 2=Two year
    lc = pd.Series(0, index=df.index)
    if 'Contract_One year' in df.columns:
        lc = lc + df['Contract_One year']
    if 'Contract_Two year' in df.columns:
        lc = lc + 2 * df['Contract_Two year']
    df['long_contract'] = lc.astype(int)
    return df

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)
print(f'Features before engineering: {X_train_basic.shape[1]}')
print(f'Features after engineering:  {X_train.shape[1]}')
print(f'New features: {[c for c in X_train.columns if c not in X_train_basic.columns]}')

In [ ]:
# Feature importance comparison (Random Forest baseline, before/after engineering)
# 'Before' = just raw encoded features, 'After' = same here but we can check importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('Top 10 features:')
print(importances.head(10))

In [ ]:
# Evaluate baseline
y_pred = rf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

In [ ]:
# Before vs After Feature Engineering — Performance Comparison
from sklearn.linear_model import LogisticRegression

# 'After' metrics: RF trained on full engineered features (cells 6-7)
after_acc = accuracy_score(y_test, rf.predict(X_test))
after_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

# 'Before' — Logistic Regression on basic encoded features only
lr_before = LogisticRegression(max_iter=1000, random_state=42)
lr_before.fit(X_train_basic, y_train)
before_acc = accuracy_score(y_test, lr_before.predict(X_test_basic))
before_auc = roc_auc_score(y_test, lr_before.predict_proba(X_test_basic)[:, 1])

print('=== Feature Engineering Impact ===')
print(f'{"Metric":<15} {"Before (LR)":<15} {"After (RF)":<15} {"Delta":<10}')
print(f'{"Accuracy":<15} {before_acc:<15.4f} {after_acc:<15.4f} {after_acc-before_acc:+.4f}')
print(f'{"ROC-AUC":<15} {before_auc:<15.4f} {after_auc:<15.4f} {after_auc-before_auc:+.4f}')
print(f'\nBefore: {X_train_basic.shape[1]} features (basic encoding only)')
print(f'After:  {X_train.shape[1]} features (+{X_train.shape[1]-X_train_basic.shape[1]} engineered)')

In [ ]:
# Save processed data for next phase
import joblib
joblib.dump(scaler, '../models/scaler.pkl')

# Save splits
pd.DataFrame(X_train).to_parquet('../data/X_train.parquet')
pd.DataFrame(X_test).to_parquet('../data/X_test.parquet')
pd.DataFrame(y_train, columns=['Churn']).to_parquet('../data/y_train.parquet')
pd.DataFrame(y_test, columns=['Churn']).to_parquet('../data/y_test.parquet')

print('Saved scaler and processed splits to data/ and models/')